In [1]:
from bs4 import BeautifulSoup, Comment
import pandas as pd

import requests
import time
import csv
from io import StringIO

import sys
from pathlib import Path

pd.set_option('display.max_columns', None)

ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
     ROOT = ROOT.parent
src_path = str(ROOT/"src")
if src_path not in sys.path:
    sys.path.insert(0,src_path)

from config import START_YEAR, END_YEAR, BASE_URL, HTTP_HEADER
import utils

### [Basketball-Reference Bot Policy](https://www.sports-reference.com/bot-traffic.html)

Basketball reference enforces a 1 day block if you are caught accessing the site more than 20 times a second.  All cells that run loops which collect the data should have `time.sleep(x)` in the loop where x $\geq3$.  In what follows we use `time.sleep(3.3)` for good measure.  Currently the cell should take about 8 minutes to run.

#### IF CHANGES HAVE BEEN MADE TO `utils.scrape_per_game_tables()` YOU NEED TO RESTART THE KERNEL, UNCOMMENT THIS CELL, AND RUN ALL.  DON'T FORGET TO RE-COMMENT OUT THIS CELL IF YOU HAVE ALREADY DOWNLOADED THE DATA AND NO CHANGES HAVE BEEN MADE TO `utils.scrape_per_game_tables()` OR YOU WILL BE SAD WHEN YOU RERUN THE NOTEBOOK!

In [ ]:
# utils.scrape_per_game_tables(START_YEAR,END_YEAR)

Scraping data for the 1980-1981 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1981-1982 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1982-1983 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1983-1984 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1984-1985 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1985-1986 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1986-1987 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1987-1988 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1988-1989 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1989-1990 season...
Adding team win/losses...
Adding MVP vote share...
Scraping data for the 1990-1991 season...
Adding team win/losses...
Ad

KeyboardInterrupt: 

when we import the datafram from the csv it is important to add a comment keyword or it will mislabel the columns using the descriptive comment instead of the names.

Example for the 2025-2026 season

In [ ]:
season = '2025-2026'
df = pd.read_csv(str(ROOT)+f'/data/{season}_season/per_game_stats_{season}.csv',encoding='utf-8',comment='#')

In [ ]:
df.sort_values('MVP', ascending=False)

,Year,Wins,Losses,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,MVP
1,2025,64.0,18.0,Shai Gilgeous-Alexander,27.0,OKC,PG,68.0,68.0,33.2,10.8,19.4,0.553,1.7,4.4,0.386,9.1,15.0,0.602,0.597,7.9,9.0,0.879,0.6,3.7,4.3,6.6,1.4,0.8,2.2,2.0,31.1,0.939
7,2025,54.0,28.0,Nikola Jokić,30.0,DEN,C,65.0,65.0,34.8,9.9,17.4,0.569,1.7,4.5,0.380,8.2,12.9,0.636,0.618,6.1,7.4,0.831,3.0,9.9,12.9,10.7,1.4,0.8,3.7,2.7,27.7,0.634
16,2025,62.0,20.0,Victor Wembanyama,22.0,SAS,C,64.0,55.0,29.2,8.6,16.9,0.512,1.9,5.5,0.349,6.7,11.4,0.590,0.569,5.8,7.0,0.827,2.0,9.5,11.5,3.1,1.0,3.1,2.4,2.4,25.0,0.569
0,2025,53.0,29.0,Luka Dončić,26.0,LAL,PG,64.0,64.0,35.8,10.8,22.8,0.476,4.0,10.8,0.366,6.9,11.9,0.575,0.563,7.9,10.1,0.780,0.6,7.1,7.7,8.3,1.6,0.5,4.0,2.4,33.5,0.250
20,2025,60.0,22.0,Cade Cunningham,24.0,DET,PG,64.0,64.0,33.9,8.6,18.6,0.461,2.0,5.7,0.342,6.6,12.9,0.514,0.513,4.8,6.0,0.812,0.9,4.7,5.5,9.9,1.4,0.8,3.7,3.1,23.9,0.117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,2025,NaN,NaN,Kevin Huerter,27.0,2TM,SF,69.0,14.0,22.4,3.9,8.6,0.451,1.4,4.7,0.308,2.4,3.9,0.624,0.535,0.8,1.0,0.775,0.5,3.0,3.5,2.6,0.9,0.5,1.0,1.7,10.0,0.000
252,2025,31.0,51.0,Kevin Huerter,27.0,CHI,SF,44.0,11.0,23.6,4.2,9.2,0.455,1.6,5.1,0.314,2.6,4.1,0.630,0.542,0.9,1.3,0.732,0.5,3.3,3.8,2.6,0.8,0.6,1.1,1.9,10.9,0.000
253,2025,60.0,22.0,Kevin Huerter,27.0,DET,SG,25.0,3.0,20.5,3.4,7.7,0.443,1.2,4.1,0.294,2.2,3.6,0.611,0.521,0.6,0.6,0.933,0.4,2.4,2.8,2.5,1.1,0.3,0.9,1.2,8.6,0.000
254,2025,42.0,40.0,Jordan Miller,26.0,LAC,SF,60.0,1.0,22.1,3.5,6.7,0.531,0.6,1.8,0.345,2.9,4.8,0.602,0.579,2.3,2.9,0.777,0.8,2.2,3.0,2.3,0.8,0.2,1.1,1.0,10.0,0.000


Note that players who have been traded during the season show up multiple times.  One for each team as well as an aggregate row where the team is listed as "#TM" and # is the number of teams they have been on. We have made the team wins/losses for these rows NaNs.  Here are the people who were on three teams.

In [ ]:
df.loc[df['Team']=='3TM']

,Year,Wins,Losses,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,MVP
428,2025,NaN,NaN,Kyle Anderson,32.0,3TM,SF,43.0,5.0,19.8,2.5,4.7,0.530,0.1,0.3,0.364,2.4,4.4,0.539,0.540,1.1,1.6,0.706,1.0,2.5,3.5,2.9,1.0,0.6,0.9,2.2,6.2,0.0
530,2025,NaN,NaN,RayJ Dennis,24.0,3TM,PG,17.0,0.0,12.1,1.5,4.9,0.313,0.8,2.6,0.311,0.7,2.2,0.316,0.398,0.5,0.5,0.889,0.5,1.1,1.5,1.9,0.2,0.2,1.0,1.2,4.4,0.0
577,2025,NaN,NaN,Patrick Baldwin Jr.,23.0,3TM,SF,9.0,1.0,11.3,1.2,3.0,0.407,0.7,1.6,0.429,0.6,1.4,0.385,0.519,0.6,0.8,0.714,1.3,1.2,2.6,0.6,0.3,0.6,0.6,1.7,3.7,0.0
618,2025,NaN,NaN,Tyus Jones,29.0,3TM,PG,67.0,12.0,14.6,1.1,3.3,0.349,0.6,2.0,0.292,0.5,1.2,0.444,0.440,0.1,0.1,0.778,0.2,1.0,1.1,2.4,0.6,0.0,0.4,0.5,3.0,0.0
641,2025,NaN,NaN,Christian Koloko,25.0,3TM,C,27.0,4.0,13.2,1.1,2.7,0.419,0.0,0.3,0.111,1.1,2.4,0.462,0.426,0.4,0.6,0.667,1.1,1.9,3.0,0.7,0.5,0.9,0.6,1.8,2.7,0.0


Now we import all of the data into a single pandas dataframe

In [ ]:
df_tot = utils.player_stat_dataframe(START_YEAR,END_YEAR)

In [ ]:
df_tot.sample(10)

,Year,Wins,Losses,Player,Age,Team,Pos,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,MVP
24723,2025,45.0,37.0,Jamaree Bouyea,26.0,PHO,PG,46.0,1.0,14.0,2.3,5.1,0.458,0.5,1.7,0.295,1.8,3.4,0.538,0.506,0.5,0.8,0.686,0.4,1.4,1.8,1.8,0.6,0.3,0.7,0.8,5.7,0.0
17449,2015,48.0,34.0,Jeff Teague,27.0,ATL,PG,79.0,78.0,28.5,5.5,12.5,0.439,1.4,3.5,0.401,4.1,9.0,0.454,0.495,3.3,3.9,0.837,0.4,2.3,2.7,5.9,1.2,0.3,2.8,2.1,15.7,0.0
15686,2012,34.0,48.0,José Calderón,31.0,TOR,PG,45.0,30.0,28.3,4.2,8.8,0.470,1.8,4.1,0.429,2.4,4.8,0.505,0.569,1.0,1.2,0.904,0.3,2.0,2.4,7.4,0.6,0.1,1.7,1.3,11.1,0.0
13457,2008,29.0,53.0,Brandan Wright,21.0,GSW,PF,39.0,23.0,17.6,3.3,6.3,0.528,0.0,0.1,0.000,3.3,6.2,0.537,0.528,1.6,2.2,0.741,1.6,2.4,4.0,0.5,0.6,0.9,0.6,1.9,8.3,0.0
14589,2010,24.0,58.0,Francisco García,29.0,SAC,SF,58.0,34.0,23.9,3.6,8.2,0.436,1.4,3.9,0.362,2.2,4.3,0.502,0.521,1.1,1.3,0.855,0.4,1.9,2.3,1.2,0.9,0.8,0.8,2.2,9.7,0.0
11208,2004,51.0,31.0,Maurice Taylor,28.0,HOU,PF,38.0,16.0,23.8,3.3,7.7,0.435,0.1,0.2,0.375,3.3,7.5,0.437,0.440,1.0,1.6,0.650,1.1,3.2,4.3,1.4,0.4,0.3,1.7,3.2,7.8,0.0
21974,2021,NaN,NaN,Chris Silva,25.0,2TM,PF,10.0,0.0,9.1,0.8,1.5,0.533,0.0,0.0,NaN,0.8,1.5,0.533,0.533,1.0,1.2,0.833,1.3,2.3,3.6,0.7,0.0,0.1,0.8,2.0,2.6,0.0
18526,2016,33.0,49.0,Manny Harris,27.0,DAL,SG,4.0,0.0,6.3,0.8,3.8,0.200,0.0,0.5,0.000,0.8,3.3,0.231,0.200,0.5,1.0,0.500,0.8,1.5,2.3,0.5,0.0,0.0,0.5,0.3,2.0,0.0
8644,1999,19.0,63.0,Adonal Foyle,24.0,GSW,C,76.0,59.0,21.8,2.5,5.0,0.508,0.0,0.0,NaN,2.5,5.0,0.508,0.508,0.4,1.2,0.378,2.3,3.3,5.6,0.6,0.3,1.8,0.9,2.9,5.5,0.0
11483,2004,44.0,38.0,Britton Johnsen,25.0,IND,SF,6.0,1.0,14.5,1.0,3.7,0.273,0.0,0.7,0.000,1.0,3.0,0.333,0.273,0.0,0.0,NaN,0.8,0.8,1.7,0.7,0.2,0.0,0.5,1.7,2.0,0.0
